# Job Interview Guide Workshop — Personalized ML Practice

**Role:** Junior ML Specialist / Data Scientist  
**Source inspiration:** CSCN8010 notebooks (Data Stream Visualization, Linear Regression Architecture, Performance Metrics Classification, KNN, Logistic Regression).  
**Interview score:** **7/15 (46.7%)**  
**Incorrect questions:** KNN k, Feature scaling,Regression tree, Classification tree,Imbalanced dataset,Validation set,R²,Overfitting,Model choice.

## How to use this notebook
1. Read each short explanation before running code.
2. Run the setup cell, then execute activities in sequence.
3. Replace `None` in each **TODO** with your own solution; the notebook remains runnable before your answers are completed.
4. Run the check cells and investigate any failed checks.
5. Complete the final reflection in your own words.

**Important:** Example datasets are generated locally with fixed random seeds. No external datasets, credentials, or database connections are needed.

## 0. Environment Setup
This exercise follows the workshop pattern of **Markdown explanation → imports → dataset inspection → implementation → evaluation → reflection**. It uses NumPy, pandas, Matplotlib and scikit-learn.

If a library is missing, uncomment the installation line below and run it once.

In [19]:
# Optional first-time installation in Jupyter (uncomment only if needed):
# %pip install numpy pandas matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.datasets import make_regression, make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import r2_score, mean_squared_error, recall_score, precision_score, f1_score, confusion_matrix

RANDOM_STATE = 42
print('Environment ready!')

Environment ready!


## 1. Training, Validation and Test Sets — Q3
**Interview concept:** Training learns model parameters; validation helps choose hyperparameters and compare candidates; the **untouched test set** measures final generalization after model decisions.

**Example:** A fictional housing dataset. We create a 60% / 20% / 20% split: train, validation and test. Use validation to choose an approach without consulting the test scores.

In [20]:
# Create a repeatable, small synthetic regression dataset.
X_housing, y_housing = make_regression(n_samples=300, n_features=3, noise=18, random_state=RANDOM_STATE)
X_dev, X_test, y_dev, y_test = train_test_split(X_housing, y_housing, test_size=0.20, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, random_state=RANDOM_STATE)
print('Train/validation/test:', len(X_train), len(X_val), len(X_test))
print('Overlap-free sizes:', len(X_train)+len(X_val)+len(X_test), '/', len(X_housing))

Train/validation/test: 180 60 60
Overlap-free sizes: 300 / 300


### Exercise 1 — TODO
Fill in the total number of validation rows and percentage of the complete dataset. Explain why we should not repeatedly use the test set to choose a model.

In [21]:
# TODO: Replace None with your answers.
validation_count = None
validation_fraction = None
# SANITY CHECK (runs when you have provided an answer)
if validation_count is not None:
    assert validation_count == len(X_val), 'Count the validation examples.'
if validation_fraction is not None:
    assert np.isclose(validation_fraction, 0.20), 'Fraction should be relative to the full dataset.'
print('Validation split exercises: checks complete or waiting for TODOs.')

Validation split exercises: checks complete or waiting for TODOs.


## 2. R² and MSE Interpretation — Q5
**Interview concept:** R² describes improvement over predicting a constant equal to the target mean. R²=0.85 means the squared error is 15% of the mean-baseline squared error on that dataset; it does **not** mean 85% accuracy or a probability of being correct.

MSE is the average squared prediction error, in squared target units. The two metrics answer different questions.

In [22]:
# Compare a learned regression model with a mean-only baseline.
reg = LinearRegression().fit(X_train, y_train)
y_pred = reg.predict(X_test)
mean_baseline = np.full(y_test.shape, y_test.mean(), dtype=float)
print('Model R²:', round(r2_score(y_test, y_pred), 3))
print('Model MSE:', round(mean_squared_error(y_test, y_pred), 2))
print('Baseline MSE:', round(mean_squared_error(y_test, mean_baseline), 2))

Model R²: 0.985
Model MSE: 245.39
Baseline MSE: 16567.06


### Exercise 2 — TODO
Compute R² yourself using the ratio of model MSE to baseline MSE. Then compare with `r2_score`. The test-set target mean is used here only to demonstrate the mathematical definition; this is **not** how we would build a deployable prediction baseline.

In [23]:
# TODO: Write an expression using the precomputed errors.
manual_r2 = None
if manual_r2 is not None:
    assert np.isclose(manual_r2, r2_score(y_test, y_pred)), 'Check how baseline and model errors relate.'
print('Your manually computed R²:', manual_r2)

Your manually computed R²: None


## 3. Overfitting and Generalization — Q6
**Interview concept:** When training performance is excellent but validation/test performance is much worse, the model may have memorized patterns or noise. A bigger learning rate is not a general fix. Compare model complexity and results on unseen data.

**Example:** Overly deep versus constrained regression trees fitted to noisy data.

In [24]:
# Show training-vs-validation performance for two model complexities.
for depth in [2, None]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE)
    tree.fit(X_train, y_train)
    print('max_depth:', depth, '| train R²:', round(tree.score(X_train,y_train),3), '| validation R²:', round(tree.score(X_val,y_val),3))

max_depth: 2 | train R²: 0.633 | validation R²: 0.544
max_depth: None | train R²: 1.0 | validation R²: 0.863


### Exercise 3 — TODO
Choose an upper bound for `max_depth` to test (such as 2, 4, 6, 10). Find the depth with the best **validation** R². Do not use the test set during this selection.

In [25]:
# TODO: Add depth candidates; fill best_depth after examining the printed results.
depth_candidates = [2, 4, 6, 10]
validation_scores = {}
for depth in depth_candidates:
    fitted = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE).fit(X_train,y_train)
    validation_scores[depth] = fitted.score(X_val,y_val)
print('Validation R² by depth:', {k:round(v,3) for k,v in validation_scores.items()})
best_depth = None
if best_depth is not None:
    assert best_depth in depth_candidates
    assert best_depth == max(validation_scores,key=validation_scores.get), 'Select using validation R².'

Validation R² by depth: {2: 0.544, 4: 0.79, 6: 0.84, 10: 0.837}


## 4. KNN Hyperparameter k — Q9
**Interview concept:** KNN uses the closest training examples. A low `k` can be very sensitive to outliers and noise; a high `k` smooths boundaries and may underfit. Select `k` using validation or cross-validation.

**Example:** Two-feature synthetic classification observations.

In [26]:
# Generate classification data and evaluate several values of k.
X_knn, y_knn = make_classification(n_samples=300, n_features=2, n_redundant=0, n_informative=2,
                                  class_sep=0.8, flip_y=0.10, random_state=RANDOM_STATE)
Xk_train, Xk_val, yk_train, yk_val = train_test_split(X_knn,y_knn,test_size=0.30,stratify=y_knn,random_state=RANDOM_STATE)
k_scores = {}
for k in [1, 3, 5, 11, 25]:
    knn_pipe = Pipeline([('scaler',StandardScaler()),('knn',KNeighborsClassifier(n_neighbors=k))])
    knn_pipe.fit(Xk_train,yk_train)
    k_scores[k]=knn_pipe.score(Xk_val,yk_val)
print(pd.Series(k_scores,name='Validation accuracy'))

1     0.844444
3     0.888889
5     0.888889
11    0.900000
25    0.877778
Name: Validation accuracy, dtype: float64


### Exercise 4 — TODO
Select the best `k` from the candidates according to the validation scores. Explain what could happen if `k=1` memorizes noisy examples.

In [27]:
best_k = None  # TODO
if best_k is not None:
    assert best_k in k_scores
    assert k_scores[best_k] == max(k_scores.values()), 'Choose a k with the highest validation accuracy.'
print('Selected k:',best_k)

Selected k: None


## 5. KNN Feature Scaling — Q10
**Interview concept:** Euclidean distance depends on feature units. Income values measured in dollars may dominate age measured in years. `KNeighborsClassifier` does not automatically scale features.

**Example:** Age and annual income; inspect how scaling changes distances.

In [28]:
# Fictional customer profiles: [age, annual_income].
customers = np.array([[25, 50000], [45, 51000], [26, 85000]],dtype=float)
raw_distance = np.linalg.norm(customers[0]-customers[1])
scaler_demo = StandardScaler().fit(customers)
scaled = scaler_demo.transform(customers)
scaled_distance = np.linalg.norm(scaled[0]-scaled[1])
print('Raw distance (row 0 to row 1):',round(raw_distance,2))
print('Scaled distance (row 0 to row 1):',round(scaled_distance,2))
print('Standardized customer profiles:\n',np.round(scaled,2))

Raw distance (row 0 to row 1): 1000.2
Scaled distance (row 0 to row 1): 2.17
Standardized customer profiles:
 [[-0.76 -0.74]
 [ 1.41 -0.68]
 [-0.65  1.41]]


### Exercise 5 — TODO
Fill in a `Pipeline` that standardizes inputs before fitting KNN. Include scaler fitting **only on training data** using `.fit(X_train, y_train)` on the pipeline.

In [29]:
# TODO: Create a pipeline with StandardScaler and KNeighborsClassifier(n_neighbors=5).
scaled_knn_pipeline = None
if scaled_knn_pipeline is not None:
    scaled_knn_pipeline.fit(Xk_train,yk_train)
    assert isinstance(scaled_knn_pipeline.named_steps['knn'],KNeighborsClassifier)
    assert isinstance(scaled_knn_pipeline.named_steps['scaler'],StandardScaler)
    print('Scaled KNN validation accuracy:',scaled_knn_pipeline.score(Xk_val,yk_val))

## 6. Regression Trees: What a Leaf Predicts — Q11
**Interview concept:** For the default squared-error criterion, a regression-tree leaf predicts the **mean** of training target values assigned to that leaf, not the maximum. Trees split input space into regions with constant predictions.

In [30]:
# Three house prices (in thousands) are in one terminal node.
leaf_prices = np.array([200., 240., 280.])
print('Training target values:',leaf_prices)
print('Mean-based leaf prediction:',np.mean(leaf_prices))
# Demonstrate the leaf prediction using a one-leaf regression tree.
leaf_reg = DecisionTreeRegressor(max_leaf_nodes=None, min_samples_split=10,random_state=RANDOM_STATE)
leaf_reg.fit(np.array([[1],[2],[3]]),leaf_prices)
print('Tree prediction:',leaf_reg.predict([[2.5]])[0])

Training target values: [200. 240. 280.]
Mean-based leaf prediction: 240.0
Tree prediction: 240.0


### Exercise 6 — TODO
Predict the price for a second leaf with target values [310, 330, 350, 370] in thousands of dollars.

In [31]:
second_leaf = np.array([310.,330.,350.,370.])
second_leaf_prediction = None  # TODO: compute the mean
if second_leaf_prediction is not None:
    assert np.isclose(second_leaf_prediction,np.mean(second_leaf))
print('Your leaf prediction:',second_leaf_prediction)

Your leaf prediction: None


## 7. Classification Trees: Labels vs. Probabilities — Q12
**Interview concept:** A classification-tree leaf usually outputs its most common class as the predicted **label**, while `predict_proba()` returns estimated class proportions. If 15 samples are Class 0 and 5 samples are Class 1, probability of Class 1 is 0.25 but the predicted class is Class 0.

**Example:** Train a tree where all samples must share one leaf.

In [32]:
# Build a one-leaf tree with 15 no-default examples and 5 default examples.
X_leaf = np.arange(20).reshape(-1,1)
y_leaf = np.array([0]*15+[1]*5)
leaf_classifier = DecisionTreeClassifier(min_samples_split=30,random_state=RANDOM_STATE)
leaf_classifier.fit(X_leaf,y_leaf)
print('Predicted class:',leaf_classifier.predict([[8]])[0])
print('Class probabilities:',dict(zip(leaf_classifier.classes_,leaf_classifier.predict_proba([[8]])[0])))

Predicted class: 0
Class probabilities: {np.int64(0): np.float64(0.75), np.int64(1): np.float64(0.25)}


### Exercise 7 — TODO
For a leaf containing 8 negative cases and 12 positive cases, give (a) the predicted class and (b) the estimated probability of the positive class.

In [33]:
new_leaf_labels = np.array([0]*8+[1]*12)
predicted_class = None   # TODO: majority label
positive_probability = None  # TODO: fraction of class 1
if predicted_class is not None:
    assert predicted_class == 1
if positive_probability is not None:
    assert np.isclose(positive_probability,0.60)
print('Your predicted class:',predicted_class,'| P(class 1):',positive_probability)

Your predicted class: None | P(class 1): None


## 8. Model Selection Under High False-Negative Cost — Q15
**Interview concept:** Choose evaluation criteria from the real business objective, not simply highest accuracy. When missing a positive case is costly, pay special attention to **recall**; also check false positives, precision, calibration, and operational constraints. A model is a **candidate**, not automatically ready for clinical use.

**Example:** Three hypothetical classifiers evaluated on the same test set.

In [34]:
# These are hypothetical, interview-style summary results, not real clinical validation.
comparison = pd.DataFrame({
 'Model':['Logistic Regression','Decision Tree','KNN'],
 'Accuracy':[0.88,0.94,0.91],
 'Recall':[0.92,0.72,0.85],
 'F1':[0.89,0.80,0.86]
})
print(comparison.to_string(index=False))
print('Highest recall candidate:',comparison.loc[comparison['Recall'].idxmax(),'Model'])

              Model  Accuracy  Recall   F1
Logistic Regression      0.88    0.92 0.89
      Decision Tree      0.94    0.72 0.80
                KNN      0.91    0.85 0.86
Highest recall candidate: Logistic Regression


### Exercise 8 — TODO
Choose the candidate for a situation where **false negatives are especially costly**, and explain in two sentences why it is better *for this objective* even if another model has higher accuracy. List two additional checks needed before deployment.

In [35]:
selected_model = None  # TODO: enter model name as a string
if selected_model is not None:
    assert selected_model == comparison.loc[comparison['Recall'].idxmax(),'Model']
print('Your model choice:',selected_model)

Your model choice: None


### Additional Sanity Check: Confusion Matrix
This is a short reinforcement of a topic you answered correctly (Q13). Verify that a system with many negatives can have high accuracy while missing all positive cases.

In [36]:
# 99 normal examples, 1 positive example: all predicted normal.
y_true_demo = np.array([0]*99+[1])
y_pred_demo = np.zeros(100,dtype=int)
print('Confusion matrix:\n',confusion_matrix(y_true_demo,y_pred_demo))
print('Recall:',recall_score(y_true_demo,y_pred_demo,zero_division=0))
print('Accuracy:',np.mean(y_true_demo==y_pred_demo))
assert recall_score(y_true_demo,y_pred_demo,zero_division=0)==0

Confusion matrix:
 [[99  0]
 [ 1  0]]
Recall: 0.0
Accuracy: 0.99


## 9. Final Student Reflection
Write in simple words, as you would explain in a junior ML Specialist interview.

1. Which of the eight missed concepts was hardest to understand, and why?
2. What is the difference between a validation dataset and a test dataset?
3. Why can a high training score be misleading?
4. What changes when KNN uses `k=1` rather than a larger k?
5. How would you explain classification-tree and regression-tree leaf predictions to a nontechnical teammate?
6. How do the business costs of false negatives change the metric you emphasize?
7. What would you do differently on your next ML project?

**My answers:**

- **1.** TODO
- **2.** TODO
- **3.** TODO
- **4.** TODO
- **5.** TODO
- **6.** TODO
- **7.** TODO

## Workshop Completion Checklist
- [ ] I can explain the purpose of train, validation and test sets.
- [ ] I can interpret R² without calling it classification accuracy.
- [ ] I can identify overfitting using training and validation performance.
- [ ] I can select k in KNN and explain why scaling matters.
- [ ] I can distinguish the class label from the probability at a tree leaf.
- [ ] I can explain why decision-tree regression uses average targets in leaves.
- [ ] I can justify model selection using recall when false negatives are costly.
- [ ] I completed the TODO exercises, sanity checks and reflection.

*Source note:* The five class workshop notebooks inspired the section format, stepwise code, commented examples, and student reflection. Synthetic datasets are new practice examples, not copied from workshop datasets.